In [ ]:
# Check if using Google Colab
try:
    from google.colab import drive

    USE_COLAB = True

    print("Running in Google Colab environment.")
except ImportError:
    USE_COLAB = False
    print("Running in local environment.")

In [ ]:
## Imports

import numpy as np
import pandas as pd
import os

import matplotlib

import datetime as dt


from dotenv import load_dotenv

try: 
    import mne
    from mne_bids import BIDSPath

    from fooof import FOOOFGroup
    from fooof.objs import fit_fooof_3d, combine_fooofs

except ImportError as err:
    if USE_COLAB:

        # mute deprecation warnings from jupyter_client.session about datetime.datetime.utcnow() being deprecated
        import warnings
        warnings.filterwarnings(
            "ignore",
            message=r".*datetime\.datetime\.utcnow\(\) is deprecated.*",
            category=DeprecationWarning,
            module=r"jupyter_client\.session",
        )
        
        print("Libraries not found. Installing libraries...")
        !pip install -U jupyter_client ipykernel jupyter_core notebook jupyterlab # install latest jupyter packages
        !pip install mne mne-bids pyvista pyvistaqt python-picard mne-icalabel pyprep fooof h5io --quiet


        import mne
        from mne_bids import BIDSPath
        from fooof import FOOOFGroup
        from fooof.objs import fit_fooof_3d, combine_fooofs

        print("Libraries loaded on colab")

    elif not USE_COLAB:
        print(f"Libraries not found on local machine: {err}")


if not USE_COLAB:
    mne.viz.set_3d_backend("pyvistaqt")
    %matplotlib qt
elif USE_COLAB:
    mne.viz.set_3d_backend("notebook")
    %matplotlib inline

In [ ]:
if USE_COLAB:
    # Access Google Drive files

    from google.colab import drive
    import os

    # get data
    GOOGLE_ROOT = "/content/drive/MyDrive/research_data/mind-wandering/Brandmeyer/"
    drive.mount("/content/drive")
    os.chdir(GOOGLE_ROOT)

    print(f"Current Directory: {os.getcwd()}")

In [ ]:
## Settings and Info

# Paths
load_dotenv()  # Load environment variables from .env file
ROOT = os.getenv("ROOT")
if USE_COLAB:
    ROOT = GOOGLE_ROOT

RAW_DATA = ROOT
DERIVATIVES = f"{ROOT}/derivatives"
DATA_FOLDER = f"{DERIVATIVES}/noica"
PREPROC_DATA = f"{DATA_FOLDER}/preprocessed"
EPOCHED_DATA = f"{DATA_FOLDER}/epoched"

# Analysis paths
ANALYSIS_PATH = f"{DATA_FOLDER}/analysis"  # Base path for all analyses
# Intermediary analysis file paths
PSD_PATH = f"{ANALYSIS_PATH}/PSD"
ZSCORE_PATH = f"{ANALYSIS_PATH}/Zscore"
FOOOF_PATH = f"{ANALYSIS_PATH}/FOOOF"

os.makedirs(ANALYSIS_PATH, exist_ok=True)  # ensure the directory exists
os.makedirs(PSD_PATH, exist_ok=True)  # ensure the directory exists
os.makedirs(ZSCORE_PATH, exist_ok=True)  # ensure the directory exists
os.makedirs(FOOOF_PATH, exist_ok=True)  # ensure the directory exists

N_JOBS = -1  # Use all available CPU cores for parallel processing

# Frequency bands
BANDS = {
    "delta": [2, 4],
    "theta": [4, 8],
    "alpha": [8, 12],
    "beta": [12, 30],
    "gamma": [30, 50],
}

SUBJECTS = [f"{i:03d}" for i in range(1, 25)]  # '001' through '024'
SESSIONS = ["01", "02", "03"]

In [ ]:
### Run FFT and save the PSD data for each subject and session
## Outputs shape (n_epochs, n_channels, n_freqs)

# Settings
l_freq = 2
h_freq = 40
window_duration = 2.0  # seconds
fft_overlap = 0.5  # 50% overlap
sampling_Freq = 256  # TODO pull this from BIDS metadata


def save_psd(session, subject):
    try:
        # Step 1a: Load the data file
        bp = BIDSPath(
            subject=subject,
            session=session,
            task="meditation",
            description="epo",  # Identifies these as epochs
            suffix="eeg",  # Official BIDS suffix
            datatype="eeg",
            root=EPOCHED_DATA,  # Point to your derivatives folder
            extension=".fif",
            check=False,  # Allows us to define custom 'desc' tags
        )
        epochs = mne.read_epochs(bp.fpath, preload=True, verbose="ERROR")

        # Step 1b: Filter the dataw
        epochs.filter(
            l_freq=l_freq,
            h_freq=h_freq,
            verbose=False,
            n_jobs=-1,  # use all CPU cores
            picks="eeg",
        )

        ## STEP 1c: Run the FFT (Welch's Method)
        # Compute power spectral density (PSD)
        n_fft = int(window_duration * sampling_Freq)  # Num of timepoints
        n_overlap = int(n_fft * fft_overlap)  # Number of overlapping samples
        psds = epochs.compute_psd(
            method="welch",
            fmin=l_freq,
            fmax=h_freq,
            n_fft=n_fft,
            n_overlap=n_overlap,
            average="mean",
            n_jobs=N_JOBS,
            verbose=False,
            picks="eeg",
        )

        # Save fooofed data
        out_bp = BIDSPath(
            subject=subject,
            session=session,
            task="meditation",
            description="epo",  # Identifies these as epochs
            suffix="psds",  # Official BIDS suffix
            datatype="eeg",
            root=PSD_PATH,  # Point to your derivatives folder
            extension=".h5",
            check=False,  # Allows us to define custom 'desc' tags
        )
        os.makedirs(out_bp.fpath.parent, exist_ok=True)  # ensure directory exists
        psds.save(fname=out_bp.fpath, overwrite=True, verbose=False)  # Save file

        mb = os.path.getsize(out_bp.fpath) // (1000**2)
        ts = dt.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        message = f"{ts} ✅ Saved PSD for Sub-{subject} Ses-{session} ({mb} MB)"
        print(message)
        return message

    except Exception as e:
        ts = dt.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        message = f"{ts} ❌ Failed on Sub-{subject} Ses-{session}: {e}"
        print(message)
        return message

    finally:
        # Make derivative dataset desc
        desc = {
            "Name": "EEG FFT Intermediary Analysis File for Mind Wandering Detection Thesis",
            "BIDSVersion": "1.8.0",
            "DatasetType": "derivative",
        }
        with open(f"{PSD_PATH}/dataset_description.json", "w") as f:
            import json

            json.dump(desc, f, indent=4)


for subject in SUBJECTS:
    for session in SESSIONS:
        save_psd(session, subject)

        # plot psd
        # bp = BIDSPath(
        #     subject=subject,
        #     session=session,
        #     task="meditation",
        #     description="epo",  # Identifies these as epochs
        #     suffix="psds",  # Official BIDS suffix
        #     datatype="eeg",
        #     root=PSD_PATH,  # Point to your derivatives folder
        #     extension=".h5",
        #     check=False,  # Allows us to define custom 'desc' tags
        # )
        # psd = mne.time_frequency.read_spectrum(bp.fpath)
        # psd.plot(average=False)

In [ ]:
%%script echo Skipping cell
# ! FOOOF implementation posponed for analysis. FOOOF-ed ML features will be implemented in 02_feature_extraction.ipynb
# ! This is to avoid the complexity of saving 3d FOOOF objects

### Run FOOOF and save the results for each subject and session

l_freq = 2
h_freq = 40
peak_width_limits = [1.0, 8.0]
max_n_peaks = 12
min_peak_height = 0.05
peak_threshold = 2
aperiodic_mode = "fixed"  # 'fixed' or 'knee'


def save_fooof(session, subject):
    try:
        ## Load the PSD data
        bp = BIDSPath(
            subject=subject,
            session=session,
            task="meditation",
            description="epo",  # Identifies these as epochs
            suffix="psd",  # Official BIDS suffix
            datatype="eeg",
            root=PSD_PATH,  # Point to your derivatives folder
            extension=".h5",
            check=False,  # Allows us to define custom 'desc' tags
        )
        psds = mne.time_frequency.read_spectrum(bp.fpath)
        psds_data = psds.get_data()  # Shape: (n_epochs, n_channels, n_freqs)
        n_epochs, n_channels, _ = psds_data.shape

        fg = FOOOFGroup(
            peak_width_limits=peak_width_limits,
            max_n_peaks=max_n_peaks,
            min_peak_height=min_peak_height,
            peak_threshold=peak_threshold,
            aperiodic_mode=aperiodic_mode,
            verbose=False,  # Suppress console output
        )

        fgs = fit_fooof_3d(
            fg=fg,
            freqs=psds.freqs,
            power_spectra=psds_data,  # Shape: (n_epochs, n_channels, n_freqs)
            freq_range=(l_freq, h_freq),
            n_jobs=-1,  # Use all available CPU cores
        )

        combined_fgs = combine_fooofs(fgs)


        raw_aperiodic = [res.aperiodic_params for res in combined_fgs]
        raw_peaks = [res.peak_params for res in combined_fgs]
        
        df = pd.DataFrame({
            'subject': subject,
            'session': session,
            'epoch_id': np.repeat(np.arange(n_epochs), n_channels),
            'channel': np.tile(psds.ch_names, n_epochs),
            'aperiodic_params': raw_aperiodic,  # Raw array per fit
            'peak_params': raw_peaks,          # Raw array per fit (n_peaks x 3)
            'r_squared': combined_fgs.get_params('r_squared'),
            'error': combined_fgs.get_params('error')
        })

        ## Save the FOOOF results
        out_bp = BIDSPath(
            subject=subject,
            session=session,
            task="meditation",
            description="epo",  # Identifies these as epochs
            suffix="fooof",  # Official BIDS suffix
            datatype="eeg",
            root=FOOOF_PATH,  # Point to your derivatives folder
            extension=".parquet",
            check=False,  # Allows us to define custom 'desc' tags
        )

        os.makedirs(out_bp.fpath.parent, exist_ok=True)
        df.to_parquet(out_bp.fpath, engine='pyarrow')

        mb = os.path.getsize(out_bp.fpath) // (1000**2)
        ts = dt.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        message = f"{ts} ✅ Saved FOOOF results for Sub-{subject} Ses-{session} ({mb} MB)"
        print(message)
        return message

    except Exception as e:
        ts = dt.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        message = f"{ts} ❌ Failed on Sub-{subject} Ses-{session}: {e}"
        print(message)
        return message

    finally:
        # Make derivative dataset desc
        desc = {
            "Name": "EEG FOOOF Intermediary Analysis File for Mind Wandering Detection Thesis",
            "BIDSVersion": "1.8.0",
            "DatasetType": "derivative",
        }
        with open(f"{FOOOF_PATH}/dataset_description.json", "w") as f:
            import json

            json.dump(desc, f, indent=4)


SESSIONS = ["01"]
SUBJECTS = ["001"]

# Run FOOOF for each subject and session
for session in SESSIONS:
    for subject in SUBJECTS:
        save_fooof(session, subject)